# Fulcher Line-Fit PDF Manual Check

Synthetic dry run for the per-frame line-fit QC PDF page. It does not need SpectroCube data or a fitted run.

In [ ]:
from pathlib import Path
import sys

repo_root = Path.cwd()
if not (repo_root / "src" / "fulcher_extractor").exists():
    repo_root = Path.cwd().parents[1]
sys.path.insert(0, str(repo_root / "src"))

preview_dir = repo_root / "local" / "qc_line_fit_pdf_manual_check"
preview_dir.mkdir(parents=True, exist_ok=True)
preview_dir

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from fulcher_extractor.extract import extract_lines
from fulcher_extractor.fit import FitConfig
from fulcher_extractor.line_database import FulcherLine
from fulcher_extractor.line_models import gaussian_area_model
from fulcher_extractor.qc import plot_line_fit_page, write_line_fit_qc
from fulcher_extractor.spectrocube_io import Spectrum

Build one synthetic frame with an isolated line, a two-component blend, and an unresolved coincident blend.

In [ ]:
sigma = 0.0273
lines = [
    FulcherLine("H2", "Q", 0, 0, "0-0", 4, 607.4827, "synthetic", "nm"),
    FulcherLine("H2", "Q", 1, 1, "1-1", 9, 623.7457, "synthetic", "nm"),
    FulcherLine("H2", "Q", 2, 2, "2-2", 3, 623.8391, "synthetic", "nm"),
    FulcherLine("H2", "Q", 1, 1, "1-1", 10, 626.2495, "synthetic", "nm"),
    FulcherLine("H2", "Q", 2, 2, "2-2", 5, 626.2495, "synthetic", "nm"),
]

wavelength = np.linspace(606.9, 626.7, 2200)
intensity = 1.0 + 0.01 * np.sin(wavelength * 3.0)
for line, area in zip(lines, [0.12, 0.08, 0.03, 0.05, 0.025]):
    intensity += gaussian_area_model(wavelength, area, line.wavelength_nm, sigma)

spectrum = Spectrum(
    source_path=repo_root / "synthetic_frame.nc",
    shot_id="synthetic",
    selectors={"frame": 0},
    wavelength_nm=wavelength,
    intensity=intensity,
    intensity_units="a.u.",
    wavelength_medium="air",
    metadata={},
)

results = extract_lines(
    spectrum,
    lines=lines,
    wavelength_min_nm=606.0,
    wavelength_max_nm=627.0,
    config=FitConfig(
        instrument_sigma_nm=sigma,
        instrument_sigma_leeway_nm=0.005,
        close_neighbor_threshold_nm=0.10,
    ),
)
[(result.line_id, result.status) for result in results]

Render the compact per-frame page in the notebook.

In [ ]:
fig = plot_line_fit_page(spectrum, results, columns=2)
plt.show()

Write the normal review PDF only.

In [ ]:
pdf_path = preview_dir / "synthetic_frame_line_fits.pdf"
written = write_line_fit_qc(spectrum, results, pdf_path=pdf_path, columns=2)
written

Turn this on when detailed individual PNGs are useful.

In [ ]:
save_individual_pngs = False

if save_individual_pngs:
    written = write_line_fit_qc(
        spectrum,
        results,
        pdf_path=preview_dir / "synthetic_frame_line_fits_with_pngs.pdf",
        individual_dir=preview_dir / "line_pngs",
        save_individual_pngs=True,
        columns=2,
    )
    written